In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
import nltk
from nltk.corpus import stopwords
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

# Exploring dataset

In [ ]:
df=pd.read_csv("CourseraDataset-Unclean.csv")

In [ ]:
df.head()

,Course Title,Rating,Level,Duration,Schedule,Review,What you will learn,Skill gain,Modules,Instructor,Offered By,Keyword,Course Url
0,Fashion as Design,4.8,Beginner level,20 hours (approximately),Flexible schedule,"2,813 reviews",NaN,"['Art History', 'Art', 'History', 'Creativity']","['Introduction', 'Heroes', 'Silhouettes', 'Cou...","['Anna Burckhardt', 'Paola Antonelli', 'Michel...",['The Museum of Modern Art'],Arts and Humanities,https://www.coursera.org/learn/fashion-design
1,Modern American Poetry,4.4,Beginner level,Approx. 34 hours to complete,Flexible schedule,100 reviews,NaN,[],"['Orientation', 'Module 1', 'Module 2', 'Modul...",['Cary Nelson'],['University of Illinois at Urbana-Champaign'],Arts and Humanities,https://www.coursera.org/learn/modern-american...
2,Pixel Art for Video Games,4.5,Beginner level,9 hours (approximately),Flexible schedule,227 reviews,NaN,[],"['Week 1: Introduction to Pixel Art', 'Week 2:...","['Andrew Dennis', 'Ricardo Guimaraes']",['Michigan State University'],Arts and Humanities,https://www.coursera.org/learn/pixel-art-video...
3,Distribución digital de la música independiente,NaN,Beginner level,Approx. 8 hours to complete,Flexible schedule,NaN,NaN,[],"['Semana 1', 'Semana 2', 'Semana 3', 'Semana 4']",['Eduardo de la Vara Brown.'],['SAE Institute México'],Arts and Humanities,https://www.coursera.org/learn/distribucion-di...
4,The Blues: Understanding and Performing an Ame...,4.8,Beginner level,Approx. 11 hours to complete,Flexible schedule,582 reviews,Students will be able to describe the blues as...,"['Music', 'Chord', 'Jazz', 'Jazz Improvisation']","['Blues Progressions – Theory and Practice ', ...",['Dariusz Terefenko'],['University of Rochester'],Arts and Humanities,https://www.coursera.org/learn/the-blues


In [ ]:
df.columns

Index(['Course Title', 'Rating', 'Level', 'Duration', 'Schedule', 'Review',
       'What you will learn', 'Skill gain', 'Modules', 'Instructor',
       'Offered By', 'Keyword', 'Course Url'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9595 entries, 0 to 9594
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Course Title         9595 non-null   object 
 1   Rating               8156 non-null   float64
 2   Level                8330 non-null   object 
 3   Duration             9333 non-null   object 
 4   Schedule             8912 non-null   object 
 5   Review               8152 non-null   object 
 6   What you will learn  4984 non-null   object 
 7   Skill gain           9595 non-null   object 
 8   Modules              9595 non-null   object 
 9   Instructor           9595 non-null   object 
 10  Offered By           9595 non-null   object 
 11  Keyword              9595 non-null   object 
 12  Course Url           9595 non-null   object 
dtypes: float64(1), object(12)
memory usage: 974.6+ KB


In [ ]:
df.shape

(9595, 13)

In [ ]:
df.isnull().sum()

,0
Course Title,0
Rating,1439
Level,1265
Duration,262
Schedule,683
Review,1443
What you will learn,4611
Skill gain,0
Modules,0
Instructor,0


In [ ]:
df.duplicated().sum()

np.int64(900)

# Data Preprocessing

In [ ]:
required_columns = [

    "Course Title",

    "Rating",

    "Level",

    "Duration",

    "Review",

    "What you will learn",

    "Skill gain",

    "Modules",

    "Instructor",

    "Offered By",

    "Keyword",

    "Course Url"
]
df = df[required_columns]

In [ ]:
df = df.rename(columns={

    "Course Title": "title",

    "Rating": "rating",

    "Level": "level",

    "Duration": "duration",

    "Review": "review",

    "What you will learn": "description",

    "Skill gain": "skills",

    "Modules": "modules",

    "Instructor": "instructor",

    "Offered By": "platform",

    "Keyword": "category",

    "Course Url": "url"
})

In [ ]:
df = df.drop_duplicates()


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.fillna("", inplace=True)

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df.shape

(8695, 12)

In [ ]:
print(df.isnull().sum())

title          0
rating         0
level          0
duration       0
review         0
description    0
skills         0
modules        0
instructor     0
platform       0
category       0
url            0
dtype: int64


In [ ]:
df.head()

,title,rating,level,duration,review,description,skills,modules,instructor,platform,category,url
0,Fashion as Design,4.8,Beginner level,20 hours (approximately),"2,813 reviews",,"['Art History', 'Art', 'History', 'Creativity']","['Introduction', 'Heroes', 'Silhouettes', 'Cou...","['Anna Burckhardt', 'Paola Antonelli', 'Michel...",['The Museum of Modern Art'],Arts and Humanities,https://www.coursera.org/learn/fashion-design
1,Modern American Poetry,4.4,Beginner level,Approx. 34 hours to complete,100 reviews,,[],"['Orientation', 'Module 1', 'Module 2', 'Modul...",['Cary Nelson'],['University of Illinois at Urbana-Champaign'],Arts and Humanities,https://www.coursera.org/learn/modern-american...
2,Pixel Art for Video Games,4.5,Beginner level,9 hours (approximately),227 reviews,,[],"['Week 1: Introduction to Pixel Art', 'Week 2:...","['Andrew Dennis', 'Ricardo Guimaraes']",['Michigan State University'],Arts and Humanities,https://www.coursera.org/learn/pixel-art-video...
3,Distribución digital de la música independiente,,Beginner level,Approx. 8 hours to complete,,,[],"['Semana 1', 'Semana 2', 'Semana 3', 'Semana 4']",['Eduardo de la Vara Brown.'],['SAE Institute México'],Arts and Humanities,https://www.coursera.org/learn/distribucion-di...
4,The Blues: Understanding and Performing an Ame...,4.8,Beginner level,Approx. 11 hours to complete,582 reviews,Students will be able to describe the blues as...,"['Music', 'Chord', 'Jazz', 'Jazz Improvisation']","['Blues Progressions – Theory and Practice ', ...",['Dariusz Terefenko'],['University of Rochester'],Arts and Humanities,https://www.coursera.org/learn/the-blues


# NLP Feature Engineering & Text Preprocessing

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
df["combined_features"] = (

    df["title"] + " " +

    df["description"] + " " +

    df["skills"] + " " +

    df["modules"] + " " +

    df["category"]
)

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
import unicodedata
import re

def clean_text(text):

    # Convert to string
    text = str(text)

    # Lowercase
    text = text.lower()

    # Normalize accented characters
    text = unicodedata.normalize('NFKD', text)

    text = text.encode('ascii', 'ignore').decode('utf-8')

    # Remove URLs
    text = re.sub(r"http\S+", "", text)

    # Remove special characters
    text = re.sub(r"[^a-zA-Z ]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # Strip spaces
    text = text.strip()

    return text

In [ ]:
df["combined_features"] = df["combined_features"].apply(clean_text)

In [ ]:
print(df[["title", "combined_features"]].head())

                                               title  \
0                                  Fashion as Design   
1                             Modern American Poetry   
2                          Pixel Art for Video Games   
3    Distribución digital de la música independiente   
4  The Blues: Understanding and Performing an Ame...   

                                   combined_features  
0  fashion as design art history art history crea...  
1  modern american poetry orientation module modu...  
2  pixel art for video games week introduction to...  
3  distribucion digital de la musica independient...  
4  the blues understanding and performing an amer...  


# TF-IDF Recommendation Engine

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

In [ ]:
tfidf_matrix = tfidf.fit_transform(df["combined_features"])

In [ ]:
similarity_matrix = cosine_similarity(tfidf_matrix)

In [ ]:
print("TF-IDF Shape:", tfidf_matrix.shape)

TF-IDF Shape: (8695, 5000)


In [ ]:
print("Similarity Matrix Shape:", similarity_matrix.shape)

Similarity Matrix Shape: (8695, 8695)


In [ ]:
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

In [ ]:
with open("tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

# Recommendation Function

In [ ]:
def recommend_courses(user_input, top_n=5):

    # Query embedding
    query_embedding = model.encode([user_input])

    # Similarity scores
    similarity_scores = cosine_similarity(

        query_embedding,

        course_embeddings
    )

    # Flatten
    similarity_scores = similarity_scores.flatten()

    # Sorted indices
    sorted_indices = similarity_scores.argsort()[::-1]

    # Store unique titles
    recommended_titles = set()

    print("\nRecommended Courses:\n")

    count = 0

    # Loop through sorted indices
    for idx in sorted_indices:

        title = df.iloc[idx]["title"]

        # Skip duplicates
        if title in recommended_titles:

            continue

        recommended_titles.add(title)

        instructor = str(df.iloc[idx]["instructor"])

        platform = str(df.iloc[idx]["platform"])

        rating = df.iloc[idx]["rating"]

        url = df.iloc[idx]["url"]

        # Clean formatting
        instructor = instructor.replace("[", "").replace("]", "").replace("'", "")

        platform = platform.replace("[", "").replace("]", "").replace("'", "")

        print("Title:", title)

        print("Instructor:", instructor)

        print("Rating:", rating)

        print("Platform:", platform)

        print("URL:", url)

        print("-" * 50)

        count += 1

        # Stop after top_n
        if count >= top_n:

            break

In [ ]:
recommend_courses("rag")

NameError: name 'model' is not defined

In [ ]:
recommend_courses("Retrieval Augmented Generation")

In [ ]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

# Generate embeddings
course_embeddings = model.encode(

    df["combined_features"].tolist(),

    show_progress_bar=True
)

print(course_embeddings.shape)

# Semantic recommendation function
# Semantic recommendation function
def recommend_courses_(user_input):

    # Query embedding
    query_embedding = model.encode([user_input])

    # Similarity
    similarity_scores = cosine_similarity(

        query_embedding,

        course_embeddings
    )

    # Flatten
    similarity_scores = similarity_scores.flatten()

    # Add similarity column
    df["similarity"] = similarity_scores

    # Convert ratings
    df["rating"] = pd.to_numeric(
        df["rating"],
        errors="coerce"
    ).fillna(0)

    # English filter
    filtered_df = df[
        df["title"].str.contains(
            r'^[A-Za-z0-9\s:,&()\-]+$',
            regex=True,
            na=False
        )
    ]

    # Similarity threshold
    filtered_df = filtered_df[
        filtered_df["similarity"] > 0.35
    ]

    # Remove duplicates
    filtered_df = filtered_df.drop_duplicates(
        subset=["title"]
    )

    # -----------------------------------
    # TOP RATED COURSES
    # -----------------------------------

    top_rated_courses = filtered_df.sort_values(

        by=["similarity", "rating"],

        ascending=False
    ).head(5)

    # Remove top rated from related list
    remaining_courses = filtered_df[
        ~filtered_df["title"].isin(
            top_rated_courses["title"]
        )
    ]

    # -----------------------------------
    # ALL RELATED COURSES
    # -----------------------------------

    all_related_courses = remaining_courses.sort_values(

        by="similarity",

        ascending=False
    ).head(20)

    return top_rated_courses, all_related_courses

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/272 [00:00<?, ?it/s]

(8695, 384)


In [ ]:
recommend_courses_("machine learning by Jon Reifschneider")

(                                                  title  rating  \
 1993  Machine Learning: Theory and Hands-on Practice...     3.7   
 1830  Machine Learning Foundations for Product Managers     4.6   
 2027         Machine Learning Introduction for Everyone     4.5   
 5034                      Machine Learning: an overview     4.4   
 2217  Machine Learning Foundations: A Case Study App...     4.6   
 
                    level                     duration          review  \
 1993  Intermediate level  3 months at 10 hours a week      39 reviews   
 1830      Beginner level     15 hours (approximately)     303 reviews   
 2027      Beginner level  Approx. 7 hours to complete     192 reviews   
 5034      Beginner level      2 hours (approximately)      41 reviews   
 2217                         18 hours (approximately)  13,358 reviews   
 
                                             description  \
 1993  Explore several classic Supervised and Unsuper...   
 1830                   

In [ ]:
with open("course_embeddings.pkl", "wb") as f:

    pickle.dump(course_embeddings, f)

In [ ]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
df.to_csv("final_courses.csv", index=False)